# Exercise P4.1: pandas Wrangling Pipeline
### STAT 540 — Week 4


## Overview

In this exercise, you will replicate the dplyr pipeline from R4.1 in pandas, then build your own analysis pipeline.

## Task 1: Replicate the R Pipeline

Translate the R4.1 Task 1 pipeline (delayed flights by carrier) into pandas:

In [3]:
import pandas as pd

flights = pd.read_csv("/content/flights.csv")   # Or use nycflights13 data

result = (
    flights
    .query("arr_delay > 0")
    .groupby("carrier")
    .agg(
        avg_delay=("arr_delay", "mean"),
        median_delay=("arr_delay", "median"),
        n=("arr_delay", "size"),
        pct_over_60=("arr_delay", lambda x: (x > 60).mean() * 100)
    )
    .sort_values("avg_delay", ascending=False)
)
print(result)

         avg_delay  median_delay      n  pct_over_60
carrier                                             
OO       60.600000          47.0     10    40.000000
YV       51.081395          27.5    258    28.682171
9E       49.272714          27.0   6637    27.572699
EV       48.268584          28.0  24484    27.785493
F9       47.579082          24.0    392    22.193878
VX       43.847079          17.0   1746    21.420389
FL       41.094459          19.0   1895    18.997361
WN       40.747549          19.0   5304    20.041478
B6       40.009064          22.0  23609    21.030116
AA       38.265552          19.0  10706    19.334952
MQ       37.852048          20.0  11693    19.866587
DL       37.743557          17.0  16413    17.833425
UA       36.650977          19.0  22222    17.689677
HA       35.030928          14.0     97     8.247423
AS       34.365079          17.0    189    17.460317
US       29.011566          15.0   7349    12.750034


**Your turn:** Compare the readability of the R and Python versions. Which do you prefer for this task and why?

> I prefer the R version, I think the csv viewer is much more detailed and allows for a closer inspection.

## Task 2: Reshape + Missing Data

In [4]:
import pandas as pd
import numpy as np

# Create data with missing values
data = pd.DataFrame({
    "student": ["Alice", "Bob", "Carol", "Dan", "Eve"],
    "math": [92, np.nan, 88, 75, np.nan],
    "english": [88, 85, np.nan, 79, 91],
    "science": [95, 72, 84, np.nan, 88]
})

# Task: Reshape to long format
long = data.melt(id_vars=["student"], var_name="subject", value_name="score")
print(long)

# Count missing
print(f"\nMissing values:\n{long['score'].isna().sum()}")

# Impute with subject mean
long["score_imputed"] = (
    long.groupby("subject")["score"]
    .transform(lambda x: x.fillna(x.mean()))
)
print(long)

   student  subject  score
0    Alice     math   92.0
1      Bob     math    NaN
2    Carol     math   88.0
3      Dan     math   75.0
4      Eve     math    NaN
5    Alice  english   88.0
6      Bob  english   85.0
7    Carol  english    NaN
8      Dan  english   79.0
9      Eve  english   91.0
10   Alice  science   95.0
11     Bob  science   72.0
12   Carol  science   84.0
13     Dan  science    NaN
14     Eve  science   88.0

Missing values:
4
   student  subject  score  score_imputed
0    Alice     math   92.0          92.00
1      Bob     math    NaN          85.00
2    Carol     math   88.0          88.00
3      Dan     math   75.0          75.00
4      Eve     math    NaN          85.00
5    Alice  english   88.0          88.00
6      Bob  english   85.0          85.00
7    Carol  english    NaN          85.75
8      Dan  english   79.0          79.00
9      Eve  english   91.0          91.00
10   Alice  science   95.0          95.00
11     Bob  science   72.0          72.00
12 

**Your turn:** Why did we impute with the subject-specific mean instead of the overall mean? When would the overall mean be more appropriate?

> The subject-specific mean makes more sense as grades are assigned according to subjects - one's performance in math isn't necessarily relevant to one's performance in science, English, or another subject. Overall mean would be relevant for a general aptitude score.

## Task 3: Readability Comparison

Take your custom analysis from R4.1 Task 2 and implement it in pandas. Compare:

In [9]:
route_delays_pandas = (
    flights
    .dropna(subset=['arr_delay']) # filter(!is.na(arr_delay))
    [['origin', 'dest', 'arr_delay']] # select(origin, dest, arr_delay)
    .assign(route=lambda df: df['origin'] + '-' + df['dest']) # mutate(route = paste(origin, dest, sep = "-"))
    .groupby('route') # group_by(route)
    .agg(
        avg_arr_delay=('arr_delay', 'mean'), # summarize(avg_arr_delay = mean(arr_delay))
        n=('arr_delay', 'size') # summarize(n = n())
    )
    .query('n >= 100') # filter(n >= 100)
    .sort_values('avg_arr_delay', ascending=True) # arrange(avg_arr_delay)
)

print("Pandas equivalent of R dplyr pipeline:")
print(route_delays_pandas.head())

Pandas equivalent of R dplyr pipeline:
         avg_arr_delay    n
route                      
EWR-SNA      -7.868227  812
JFK-HNL      -6.915205  342
JFK-STT      -6.372727  330
EWR-EGE      -5.349057  106
LGA-DAY      -4.947368  342


**Your turn:** Fill in this comparison for YOUR specific analysis:

| Aspect | R (dplyr) | Python (pandas) |
|--------|-----------|-----------------|
| Lines of code | 10 | 15 |
| Readability (1-5) | 5 | 3 |
| What was easier | Calling and using the function | Filtering out NA values |
| What was harder | Mutating the route column | Recalling function names |

## Submission

```bash
git add week04/exercises/P4.1*
git commit -m "Complete Exercise P4.1: pandas wrangling pipeline"
git push origin main
```